<a href="https://colab.research.google.com/github/TipsyPanda/ComplexBridges/blob/main/ECG_MultiLabel_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ECG Multi-Label Model Training (Self-Contained)
Fully independent notebook: loads preprocessed data, trains multi-label model, saves results

In [7]:
!pip install tensorflow scikit-learn matplotlib seaborn huggingface-hub h5py -q


import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import hamming_loss, precision_score, recall_score, f1_score

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Environment & Data Loading

In [9]:
# Download preprocessed data from Hugging Face
from huggingface_hub import snapshot_download
import h5py

print("Downloading preprocessed data from Hugging Face...")
local_dir = snapshot_download(
    repo_id="kiril-buga/ECG-database",
    repo_type="dataset",
    local_dir="./ECG-database/",
    allow_patterns=["multilabel_v2/*"]  # Only download preprocessed data (~1.5-2 GB)
)

DATA_DIR = os.path.join(local_dir, "multilabel_v2")
RESULTS_DIR = "./results/"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"✓ Downloaded to: {DATA_DIR}")
print(f"✓ Results will be saved to: {RESULTS_DIR}")

# ============================================================================
# Load dataset structure from HDF5 and create streaming data generator
h5_path = os.path.join(DATA_DIR, "ecg_data.h5")

print(f"\nOpening HDF5 file: {h5_path}")
with h5py.File(h5_path, 'r') as h5f:
    n_samples = h5f['X'].shape[0]
    input_shape = h5f['X'].shape[1:]
    DISEASE_CLASSES = list(h5f.attrs['disease_classes'])
    data_format = h5f.attrs.get('data_format', 'unknown')

print(f"✓ Dataset info:")
print(f"  Total windows: {n_samples}")
print(f"  Input shape: {input_shape}")
print(f"  Data format: {data_format}")
print(f"  Classes: {DISEASE_CLASSES}")

# Create train/val/test split indices (without loading data)
all_idx = np.arange(n_samples)
train_idx, test_idx = train_test_split(all_idx, test_size=0.2, random_state=42)
train_idx, val_idx = train_test_split(train_idx, test_size=0.25, random_state=42)

print(f"\nTrain: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# Create tf.data.Dataset objects
def create_dataset(h5_path, indices, batch_size=32, is_training=True):
    """Create streaming dataset from HDF5.

    NOTE: HDF5 requires indices in increasing order for fancy indexing.
    We sort indices before access and unsort after to maintain shuffling.
    """
    def gen():
        with h5py.File(h5_path, 'r') as h5f:
            X_dset = h5f['X']
            y_dset = h5f['y']

            # Shuffle for training, don't shuffle for val/test
            idx = np.random.permutation(indices) if is_training else indices

            for i in range(0, len(idx), batch_size):
                batch_idx = idx[i:i+batch_size]

                # HDF5 requires sorted indices for fancy indexing
                # So we sort, access, then unsort to maintain shuffle
                batch_idx_sorted = np.sort(batch_idx)
                X_batch = X_dset[batch_idx_sorted]
                y_batch = y_dset[batch_idx_sorted]

                # Create unsort permutation to restore original shuffle order
                unsort_idx = np.argsort(np.argsort(batch_idx))
                X_batch = X_batch[unsort_idx]
                y_batch = y_batch[unsort_idx]

                # Convert float16 to float32 if needed
                if X_batch.dtype == np.float16:
                    X_batch = X_batch.astype(np.float32)

                yield X_batch, y_batch

    dataset = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(None, *input_shape), dtype=tf.float32),
            tf.TensorSpec(shape=(None, len(DISEASE_CLASSES)), dtype=tf.int32)
        )
    )

    return dataset.prefetch(tf.data.AUTOTUNE)

# Create datasets
print("\nCreating streaming datasets...")
BATCH_SIZE = 32
train_dataset = create_dataset(h5_path, train_idx, batch_size=BATCH_SIZE, is_training=True)
val_dataset = create_dataset(h5_path, val_idx, batch_size=BATCH_SIZE, is_training=False)
test_dataset = create_dataset(h5_path, test_idx, batch_size=BATCH_SIZE, is_training=False)

print(f"✓ Datasets created (batch_size={BATCH_SIZE})")

# Verify dataset contents
print("\nVerifying datasets...")
for batch_X, batch_y in train_dataset.take(1):
    print(f"  Sample batch - X shape: {batch_X.shape}, y shape: {batch_y.shape}")
    for i, cls in enumerate(DISEASE_CLASSES):
        count = int(tf.reduce_sum(batch_y[:, i]).numpy())
        print(f"    {cls}: {count} in batch")
    print("✓ Dataset verification successful!")

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Downloaded to: /content/ECG-database/multilabel_v2
✓ Results will be saved to: ./results/

Opening HDF5 file: /content/ECG-database/multilabel_v2/ecg_data.h5
✓ Dataset info:
  Total windows: 59944
  Input shape: (5000, 12)
  Data format: unknown
  Classes: ['Myocarditis', 'Cardiomyopathy', 'Kawasaki', 'CHD', 'Healthy']

Train: 35966, Val: 11989, Test: 11989

Creating streaming datasets...
✓ Datasets created (batch_size=32)

Verifying datasets...
  Sample batch - X shape: (32, 5000, 12), y shape: (32, 5)
    Myocarditis: 1 in batch
    Cardiomyopathy: 0 in batch
    Kawasaki: 0 in batch
    CHD: 1 in batch
    Healthy: 30 in batch
✓ Dataset verification successful!


## Build & Train Model

In [11]:
# Build multi-label 1D CNN model
def build_model(input_shape, num_classes):
    """Build 1D CNN for multi-label classification."""
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(32, 3, padding='same', activation='relu'),
        layers.Conv1D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),
        layers.Conv1D(64, 3, padding='same', activation='relu'),
        layers.Conv1D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),
        layers.Conv1D(128, 3, padding='same', activation='relu'),
        layers.Conv1D(128, 3, padding='same', activation='relu'),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),
        layers.Conv1D(256, 3, padding='same', activation='relu'),
        layers.Conv1D(256, 3, padding='same', activation='relu'),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),
        layers.GlobalAveragePooling1D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='sigmoid')  # Multi-label: sigmoid
    ])

print("Building model...")
model = build_model(input_shape, len(DISEASE_CLASSES))

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',  # Multi-label loss
    metrics=['binary_accuracy']
)

print("✓ Model built")
model.summary()

Building model...
✓ Model built


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 5000, 32)       │         1,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 5000, 32)       │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 2500, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2500, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 2500, 64)       │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 2500, 64)       │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 1250, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1250, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 1250, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 1250, 128)      │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 625, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 625, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 625, 256)       │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 625, 256)       │       196,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 312, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 312, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 491,589 (1.88 MB)

 Trainable params: 491,589 (1.88 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = os.path.join(RESULTS_DIR, f"model_{timestamp}.keras")

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint(model_path, monitor='val_binary_accuracy', save_best_only=True)
]

print("Training with streaming data...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=100,
    callbacks=callbacks,
    verbose=1
)
print(f"✓ Model saved: {model_path}")

Training with streaming data...
Epoch 1/100
   1124/Unknown 135s 107ms/step - binary_accuracy: 0.9434 - loss: 0.2016

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1124/1124 ━━━━━━━━━━━━━━━━━━━━ 172s 140ms/step - binary_accuracy: 0.9434 - loss: 0.2016 - val_binary_accuracy: 0.9454 - val_loss: 0.1782 - learning_rate: 0.0010
Epoch 2/100
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 138s 123ms/step - binary_accuracy: 0.9452 - loss: 0.1798 - val_binary_accuracy: 0.9454 - val_loss: 0.1740 - learning_rate: 0.0010
Epoch 3/100
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 134s 119ms/step - binary_accuracy: 0.9475 - loss: 0.1715 - val_binary_accuracy: 0.9454 - val_loss: 0.1716 - learning_rate: 0.0010
Epoch 4/100
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - binary_accuracy: 0.9471 - loss: 0.1673

KeyboardInterrupt: 

# Evaluate on test dataset
print("Evaluating on test set...")
y_test_list = []
y_pred_list = []
y_probs_list = []

for X_batch, y_batch in test_dataset:
    y_test_list.append(y_batch.numpy())
    y_probs = model.predict(X_batch, verbose=0)
    y_probs_list.append(y_probs)
    y_pred = (y_probs >= 0.5).astype(int)
    y_pred_list.append(y_pred)

# Concatenate all batches
y_test = np.concatenate(y_test_list, axis=0)
y_pred = np.concatenate(y_pred_list, axis=0)
y_pred_probs = np.concatenate(y_probs_list, axis=0)

print("\n" + "="*70)
print("TEST SET METRICS")
print("="*70)

print(f"\nHamming Loss: {hamming_loss(y_test, y_pred):.4f}")
print(f"Exact Match Accuracy: {np.mean(np.all(y_test == y_pred, axis=1)):.4f}")

print(f"\nPer-Class Metrics:")
print(f"{'Class':<20} {'Precision':>12} {'Recall':>12} {'F1-Score':>12}")
print("-" * 70)

for i, cls in enumerate(DISEASE_CLASSES):
    p = precision_score(y_test[:, i], y_pred[:, i], zero_division=0)
    r = recall_score(y_test[:, i], y_pred[:, i], zero_division=0)
    f = f1_score(y_test[:, i], y_pred[:, i], zero_division=0)
    print(f"{cls:<20} {p:>12.4f} {r:>12.4f} {f:>12.4f}")

## Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['binary_accuracy'], label='Train', linewidth=2)
axes[1].plot(history.history['val_binary_accuracy'], label='Val', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'training_{timestamp}.png'), dpi=150, bbox_inches='tight')
print(f"✓ Plot saved")
plt.show()

## Save Results

In [ ]:
# Save predictions
np.savez(
    os.path.join(RESULTS_DIR, f"predictions_{timestamp}.npz"),
    y_true=y_test,
    y_pred=y_pred,
    y_probs=y_pred_probs
)

# Save results JSON
results = {
    'timestamp': timestamp,
    'disease_classes': DISEASE_CLASSES,
    'epochs_trained': len(history.history['loss']),
    'train_samples': X_train.shape[0],
    'val_samples': X_val.shape[0],
    'test_samples': X_test.shape[0],
}

with open(os.path.join(RESULTS_DIR, f"results_{timestamp}.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"✓ Results saved to {RESULTS_DIR}")